# Extract Yearly OSM POIs

Extract routing-relevant infrastructure points from each historical Austria PBF and save one normalized GeoParquet per year in `TOOLS/osm-data/`.

The category logic follows `TOOLS/osm-data/geofabrik-osm-gis-standard-0.7.pdf`: motorway junctions, fuel, parking, railway stations/halts, and road classes used as federal-road candidates.

In [1]:
import re
import warnings
from pathlib import Path

import geopandas as gpd
import pandas as pd
import pyogrio

def find_project_dir(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "README.md").exists() and (path / "ANAL").exists() and (path / "TOOLS").exists():
            return path
    raise FileNotFoundError("Could not find project root")


PROJECT_DIR = find_project_dir(Path.cwd())
OSM_DIR = PROJECT_DIR / "TOOLS" / "osm-data"
GIP_DIR = PROJECT_DIR / "TOOLS" / "gip-data"
ROUTING_DIR = PROJECT_DIR / "TOOLS" / "routing"
MUNICIPALITIES_PATH = PROJECT_DIR / "OGD" / "Gemeindegrenzen.zip"
BILDUNGSSTANDORTE_PATH = PROJECT_DIR / "OGD" / "Bildungsstandorte.zip"
HALTESTELLEN_PATH = PROJECT_DIR / "OGD" / "Haltestellen.zip"
STATIC_DESTINATIONS_PATH = ROUTING_DIR / "static_routing_destinations.csv"

CRS_ANALYSIS = "EPSG:3035"
CRS_OSM = "EPSG:4326"
YEARS = range(2015, 2026)

PBF_BY_YEAR = {year: GIP_DIR / f"{year}.osm.pbf" for year in YEARS}
POI_OUTPUT_BY_YEAR = {year: OSM_DIR / f"austria-{year}-pois.geoparquet" for year in YEARS}


## POI mapping

The raw PBF is read through GDAL's OSM layers. It does not expose Geofabrik's `fclass` column directly, so the notebook applies equivalent OSM tag filters and writes a normalized `poi_type` field.

In [2]:
POINT_TAGS = {
    "motorway_exit": [("highway", {"motorway_link"})],
    "rail_station": [("railway", {"station"})],
}

# `motorway_link` is stored in the raw PBF `lines` layer, not the `points` layer.
# Keep the temporary POINT_TAGS definition above, but derive these POIs from line midpoints.
LINE_DERIVED_POI_TYPES = {"motorway_exit"}

COMMON_COLUMNS = [
    "osm_id",
    "name",
    "highway",
    "railway",
    "amenity",
    "ref",
    "other_tags",
    "geometry",
]

NORMALIZED_POI_COLUMNS = COMMON_COLUMNS + [
    "year",
    "poi_type",
    "poi_id",
    "source_poi_id",
    "source_file",
    "source_schema",
    "source_note",
    "pt_departures_weekday",
    "static_destination",
]


In [3]:
def read_zipped_shapefile(zip_path: Path) -> gpd.GeoDataFrame:
    archive_path = zip_path.resolve().as_posix()
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*FLAECHE_HA parsed incompletely.*")
        return gpd.read_file(f"zip://{archive_path}")


def styria_buffer_wgs84(buffer_m: int = 5000):
    municipalities = read_zipped_shapefile(MUNICIPALITIES_PATH).to_crs(CRS_ANALYSIS)
    boundary = municipalities.union_all().buffer(buffer_m)
    boundary_wgs84 = gpd.GeoSeries([boundary], crs=CRS_ANALYSIS).to_crs(CRS_OSM)
    return boundary_wgs84.iloc[0]


def keep_existing_columns(frame: gpd.GeoDataFrame, columns: list[str]) -> gpd.GeoDataFrame:
    existing = [column for column in columns if column in frame.columns]
    return frame[existing].copy()


def align_poi_columns(frame: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    for column in NORMALIZED_POI_COLUMNS:
        if column not in frame.columns:
            frame[column] = pd.NA
    return frame[NORMALIZED_POI_COLUMNS].copy()


def has_other_tag(value: object, names: tuple[str, ...]) -> bool:
    if not isinstance(value, str):
        return False
    lowered = value.lower()
    return any(name in lowered for name in names)


def other_tags_match(value: object, key: str, allowed_values: set[str]) -> bool:
    if not isinstance(value, str):
        return False
    lowered = value.lower()
    return any(f'"{key.lower()}"=>"{allowed.lower()}"' in lowered for allowed in allowed_values)


def extract_other_tag(value: object, key: str) -> str | None:
    if not isinstance(value, str):
        return None
    match = re.search(rf'"{re.escape(key)}"=>"([^"]*)"', value)
    return match.group(1) if match else None


def fill_ref_from_other_tags(frame: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if "other_tags" not in frame.columns:
        return frame
    extracted_ref = frame["other_tags"].apply(lambda value: extract_other_tag(value, "ref"))
    if "ref" in frame.columns:
        frame["ref"] = frame["ref"].fillna(extracted_ref)
    else:
        frame["ref"] = extracted_ref
    return frame


def tag_filter_match(frame: gpd.GeoDataFrame, key: str, allowed_values: set[str]) -> pd.Series:
    selected = pd.Series(False, index=frame.index)
    if key in frame.columns:
        selected = selected | frame[key].isin(allowed_values)
    if "other_tags" in frame.columns:
        selected = selected | frame["other_tags"].apply(lambda value: other_tags_match(value, key, allowed_values))
    return selected


def normalize_pois(frame: gpd.GeoDataFrame, year: int, poi_type: str) -> gpd.GeoDataFrame:
    frame = fill_ref_from_other_tags(frame)
    frame = keep_existing_columns(frame, COMMON_COLUMNS)
    frame["year"] = year
    frame["poi_type"] = poi_type
    frame["source_poi_id"] = frame["osm_id"].astype("string") if "osm_id" in frame.columns else pd.NA
    frame["source_file"] = PBF_BY_YEAR[year].name
    frame["source_schema"] = "raw_osm_pbf_gdal_filters_geofabrik_standard_0_7"
    frame["source_note"] = "historical_osm"
    frame["static_destination"] = False
    return align_poi_columns(frame)


def extract_point_pois(pbf_path: Path, year: int, mask: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    points = pyogrio.read_dataframe(pbf_path, layer="points", mask=mask)
    output = []
    for poi_type, tag_filters in POINT_TAGS.items():
        if poi_type in LINE_DERIVED_POI_TYPES:
            continue
        selected = pd.Series(False, index=points.index)
        for column, allowed_values in tag_filters:
            selected = selected | tag_filter_match(points, column, allowed_values)
        subset = points[selected].copy()
        if not subset.empty:
            output.append(normalize_pois(subset, year, poi_type))
    return pd.concat(output, ignore_index=True) if output else gpd.GeoDataFrame(columns=NORMALIZED_POI_COLUMNS, geometry="geometry", crs=CRS_OSM)


def extract_line_derived_pois(pbf_path: Path, year: int, mask) -> gpd.GeoDataFrame:
    lines = pyogrio.read_dataframe(pbf_path, layer="lines", mask=mask)
    output = []
    for poi_type in LINE_DERIVED_POI_TYPES:
        tag_filters = POINT_TAGS[poi_type]
        selected = pd.Series(False, index=lines.index)
        for column, allowed_values in tag_filters:
            selected = selected | tag_filter_match(lines, column, allowed_values)
        subset = lines[selected].copy()
        if subset.empty:
            continue
        subset_3035 = subset.to_crs(CRS_ANALYSIS)
        subset_3035["geometry"] = subset_3035.geometry.interpolate(0.5, normalized=True)
        subset_points = subset_3035.to_crs(CRS_OSM)
        output.append(normalize_pois(subset_points, year, poi_type))
    return pd.concat(output, ignore_index=True) if output else gpd.GeoDataFrame(columns=NORMALIZED_POI_COLUMNS, geometry="geometry", crs=CRS_OSM)


def extract_higher_education_pois(year: int, mask) -> gpd.GeoDataFrame:
    if not BILDUNGSSTANDORTE_PATH.exists():
        return gpd.GeoDataFrame(columns=NORMALIZED_POI_COLUMNS, geometry="geometry", crs=CRS_OSM)

    education = read_zipped_shapefile(BILDUNGSSTANDORTE_PATH).to_crs(CRS_OSM)
    higher_ed_types = {"Universität", "Fachhochschule", "Pädagogische Hochschule"}
    selected = education[
        education["EINRICHTUN"].isin(higher_ed_types)
        | education["TYP_LANG"].isin(higher_ed_types)
        | education["TYP_DETAIL"].isin(higher_ed_types)
    ].copy()
    if selected.empty:
        return gpd.GeoDataFrame(columns=NORMALIZED_POI_COLUMNS, geometry="geometry", crs=CRS_OSM)
    selected = selected[selected.geometry.within(mask)].copy()
    selected["osm_id"] = pd.NA
    selected["name"] = selected["NAME"]
    selected["highway"] = pd.NA
    selected["railway"] = pd.NA
    selected["amenity"] = "higher_education"
    selected["ref"] = selected["KENNZAHL"].astype("string")
    selected["other_tags"] = selected.apply(
        lambda row: f'EINRICHTUN={row.get("EINRICHTUN", "")};TYP_DETAIL={row.get("TYP_DETAIL", "")};ORT={row.get("ORT", "")}',
        axis=1,
    )
    selected["year"] = year
    selected["poi_type"] = "higher_education"
    selected["source_poi_id"] = selected["OBJECTID"].astype("string")
    selected["source_file"] = BILDUNGSSTANDORTE_PATH.name
    selected["source_schema"] = "ogd_bildungsstandorte_higher_education"
    selected["source_note"] = "static_ogd_destination"
    selected["static_destination"] = True
    return align_poi_columns(selected)


def extract_pt_stop_pois(year: int, mask) -> gpd.GeoDataFrame:
    if not HALTESTELLEN_PATH.exists():
        return gpd.GeoDataFrame(columns=NORMALIZED_POI_COLUMNS, geometry="geometry", crs=CRS_OSM)

    stops = read_zipped_shapefile(HALTESTELLEN_PATH).to_crs(CRS_OSM)
    selected = stops[stops.geometry.within(mask)].copy()
    if selected.empty:
        return gpd.GeoDataFrame(columns=NORMALIZED_POI_COLUMNS, geometry="geometry", crs=CRS_OSM)

    weekday_departures = selected[["MoFr_S", "MoFr_F"]].apply(pd.to_numeric, errors="coerce")
    selected["osm_id"] = pd.NA
    selected["name"] = selected["HNAME_LANG"].fillna(selected["HNAME_KURZ"])
    selected["highway"] = pd.NA
    selected["railway"] = pd.NA
    selected["amenity"] = "public_transport_stop"
    selected["ref"] = selected["HNR"].astype("string")
    selected["other_tags"] = selected.apply(
        lambda row: ";".join([
            f'LINIE={"" if pd.isna(row.get("LINIE")) else row.get("LINIE")}',
            f'MoFr_S={"" if pd.isna(row.get("MoFr_S")) else row.get("MoFr_S")}',
            f'MoFr_F={"" if pd.isna(row.get("MoFr_F")) else row.get("MoFr_F")}',
            f'Abfahrten={"" if pd.isna(row.get("Abfahrten")) else row.get("Abfahrten")}',
        ]),
        axis=1,
    )
    selected["year"] = year
    selected["poi_type"] = "pt_stop"
    selected["source_poi_id"] = selected["HNR"].astype("string")
    selected["source_file"] = HALTESTELLEN_PATH.name
    selected["source_schema"] = "ogd_haltestellen_verkehrsverbund_steiermark"
    selected["source_note"] = "static_ogd_destination"
    selected["pt_departures_weekday"] = weekday_departures.mean(axis=1)
    selected["static_destination"] = True
    return align_poi_columns(selected)


def extract_curated_centre_pois(year: int, mask) -> gpd.GeoDataFrame:
    if not STATIC_DESTINATIONS_PATH.exists():
        return gpd.GeoDataFrame(columns=NORMALIZED_POI_COLUMNS, geometry="geometry", crs=CRS_OSM)

    centres = pd.read_csv(STATIC_DESTINATIONS_PATH)
    required = {"poi_type", "name", "municipality_name"}
    missing = sorted(required - set(centres.columns))
    if missing:
        raise ValueError(f"{STATIC_DESTINATIONS_PATH} is missing required columns: {missing}")

    municipalities = read_zipped_shapefile(MUNICIPALITIES_PATH).to_crs(CRS_ANALYSIS)
    centre_points = municipalities[["GEMNR", "GEMNAM", "geometry"]].copy()
    centre_points["geometry"] = centre_points.geometry.representative_point()
    merged = centres.merge(centre_points, left_on="municipality_name", right_on="GEMNAM", how="left", validate="many_to_one")
    missing_municipalities = sorted(merged.loc[merged["geometry"].isna(), "municipality_name"].dropna().unique())
    if missing_municipalities:
        raise ValueError(f"Unknown municipality names in {STATIC_DESTINATIONS_PATH}: {missing_municipalities}")

    output = gpd.GeoDataFrame(merged, geometry="geometry", crs=CRS_ANALYSIS).to_crs(CRS_OSM)
    output = output[output.geometry.within(mask)].copy()
    output["osm_id"] = pd.NA
    output["highway"] = pd.NA
    output["railway"] = pd.NA
    output["amenity"] = pd.NA
    output["ref"] = output["GEMNR"].astype("string")
    output["other_tags"] = "municipality_name=" + output["municipality_name"].astype("string")
    output["year"] = year
    output["source_poi_id"] = output["poi_type"].astype("string") + "_" + output["municipality_name"].astype("string")
    output["source_file"] = STATIC_DESTINATIONS_PATH.name
    output["source_schema"] = "curated_static_municipality_representative_points"
    if "source_note" not in output.columns:
        output["source_note"] = "curated_static_destination"
    output["static_destination"] = True
    return align_poi_columns(output)


def extract_static_pois(year: int, mask) -> gpd.GeoDataFrame:
    static_tables = [
        extract_higher_education_pois(year, mask),
        extract_pt_stop_pois(year, mask),
        extract_curated_centre_pois(year, mask),
    ]
    static_tables = [table for table in static_tables if not table.empty]
    if not static_tables:
        return gpd.GeoDataFrame(columns=NORMALIZED_POI_COLUMNS, geometry="geometry", crs=CRS_OSM)
    return gpd.GeoDataFrame(pd.concat(static_tables, ignore_index=True), geometry="geometry", crs=CRS_OSM)


def extract_year_pois(year: int, overwrite: bool = False) -> Path:
    pbf_path = PBF_BY_YEAR[year]
    output_path = POI_OUTPUT_BY_YEAR[year]
    if output_path.exists() and not overwrite:
        print(f"Skip {year}: {output_path.name} already exists")
        return output_path
    if not pbf_path.exists():
        raise FileNotFoundError(pbf_path)

    mask = styria_buffer_wgs84()
    point_pois = extract_point_pois(pbf_path, year, mask)
    line_pois = extract_line_derived_pois(pbf_path, year, mask)
    static_pois = extract_static_pois(year, mask)
    pois = pd.concat([point_pois, line_pois, static_pois], ignore_index=True)
    pois = gpd.GeoDataFrame(pois, geometry="geometry", crs=CRS_OSM).to_crs(CRS_ANALYSIS)
    fallback_id = pd.Series(pois.index, index=pois.index).astype("string")
    source_id = pois["source_poi_id"].astype("string").fillna(fallback_id)
    source_id = source_id.str.replace(r"[^0-9A-Za-z_]+", "_", regex=True).str.strip("_")
    pois["poi_id"] = pois["poi_type"].astype("string") + "_" + pois["year"].astype("string") + "_" + source_id
    pois = align_poi_columns(pois)
    pois.to_parquet(output_path, index=False)
    print(f"Wrote {len(pois):,} POIs for {year} to {output_path}")
    return output_path


In [4]:
# Set OVERWRITE = True to rebuild existing yearly POI files.
OVERWRITE = True

for year in YEARS:
    extract_year_pois(year, overwrite=OVERWRITE)

Wrote 8,715 POIs for 2015 to D:\CO2_Masterarbeit\CO2_Masterarbeit\TOOLS\osm-data\austria-2015-pois.geoparquet
Wrote 8,742 POIs for 2016 to D:\CO2_Masterarbeit\CO2_Masterarbeit\TOOLS\osm-data\austria-2016-pois.geoparquet
Wrote 8,749 POIs for 2017 to D:\CO2_Masterarbeit\CO2_Masterarbeit\TOOLS\osm-data\austria-2017-pois.geoparquet
Wrote 8,838 POIs for 2018 to D:\CO2_Masterarbeit\CO2_Masterarbeit\TOOLS\osm-data\austria-2018-pois.geoparquet
Wrote 8,867 POIs for 2019 to D:\CO2_Masterarbeit\CO2_Masterarbeit\TOOLS\osm-data\austria-2019-pois.geoparquet
Wrote 8,925 POIs for 2020 to D:\CO2_Masterarbeit\CO2_Masterarbeit\TOOLS\osm-data\austria-2020-pois.geoparquet
Wrote 8,942 POIs for 2021 to D:\CO2_Masterarbeit\CO2_Masterarbeit\TOOLS\osm-data\austria-2021-pois.geoparquet
Wrote 9,018 POIs for 2022 to D:\CO2_Masterarbeit\CO2_Masterarbeit\TOOLS\osm-data\austria-2022-pois.geoparquet
Wrote 9,018 POIs for 2023 to D:\CO2_Masterarbeit\CO2_Masterarbeit\TOOLS\osm-data\austria-2023-pois.geoparquet
Wrote 8,97

In [5]:
summary_records = []
for year, path in POI_OUTPUT_BY_YEAR.items():
    if path.exists():
        pois = gpd.read_parquet(path)
        counts = pois.groupby("poi_type").size().reset_index(name="count")
        counts["year"] = year
        summary_records.append(counts)

poi_summary = pd.concat(summary_records, ignore_index=True) if summary_records else pd.DataFrame()
poi_summary

,poi_type,count,year
0,higher_education,9,2015
1,motorway_exit,1094,2015
2,pt_stop,7435,2015
3,rail_station,145,2015
4,regional_centre,13,2015
...,...,...,...
61,motorway_exit,1375,2025
62,pt_stop,7435,2025
63,rail_station,135,2025
64,regional_centre,13,2025
